In [1]:
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt

In [12]:
df = pd.read_csv('Analisis GAC Full7agosto 2026(Bitacora de Piso).csv', encoding='latin-1')

In [15]:
df.columns = df.columns.str.strip()  # quitar espacios al final de nombres


In [17]:
df.info

<bound method DataFrame.info of        ï»¿Fecha          Nombre Cliente  Telefono    Asesor Esatus Lead  \
0    04/01/2025  Ubaldo Cruz Hernandez        NaN   Aurelio      Activo   
1    04/01/2025          Olivia Cabello       NaN     Pedro      Activo   
2    04/02/2025      Ana Laura Espinoza       NaN  Mauricio      Activo   
3    04/02/2025          Elvis Sanchez        NaN   Aurelio      Activo   
4    04/03/2025  Ariel Fernando LÃ³pez        NaN     Alan       Activo   
..          ...                     ...       ...       ...         ...   
229         NaN                     NaN       NaN       NaN         NaN   
230         NaN                     NaN       NaN       NaN         NaN   
231         NaN                     NaN       NaN       NaN         NaN   
232         NaN                     NaN       NaN       NaN         NaN   
233         NaN                     NaN       NaN       NaN         NaN   

       PDM    SDC Estatus de SDC  Venta        Temperatura  ... Unn

In [19]:

# --- Asesor ---
df['Asesor'] = df['Asesor'].astype(str).str.strip()
df['Asesor'] = df['Asesor'].replace('nan', 'No asignado')
asesor_map = {
    'Aurelio Torres': 'Aurelio', 'Alan González': 'Alan', 'Alan Gonzalez': 'Alan',
    'Pedro González': 'Pedro', 'Pedro Antonio': 'Pedro', 'Pedro Antonio Cinto': 'Pedro',
    'Victor Manuel Sanchez': 'Victor', 'Victor Sanchez': 'Victor', 'Victor Manuel': 'Victor',
    'Ricardo Silva': 'Ricardo', 'Mauricio Vazquez': 'Mauricio', 'Mauricio Vázquez': 'Mauricio',
    'Nancy OLascoaga': 'Nancy', 'AUrelio': 'Aurelio', 'Marcos Coca': 'Marcos',
    'Miguel Ramírez': 'Miguel', 'Casa': 'Casa/Interno'
}
df['Asesor'] = df['Asesor'].replace(asesor_map)



In [20]:

df['Esatus Lead'] = df['Esatus Lead'].astype(str).str.strip()
df['Esatus Lead'] = df['Esatus Lead'].replace('nan', 'No especificado')
df['Esatus Lead'] = df['Esatus Lead'].str.title()

In [21]:
# --- Temperatura ---
df['Temperatura'] = df['Temperatura'].astype(str).str.strip()
df['Temperatura'] = df['Temperatura'].replace('nan', 'No registrada')


In [23]:
# --- Booleanas a categóricas ---
for col in ['PDM', 'SDC', 'Venta']:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'True': 'Sí', 'False': 'No', 'nan': 'No registrado'})


df['Inter Gerente'] = df['Inter Gerente'].astype(str).str.strip()
df['Inter Gerente'] = df['Inter Gerente'].replace({'True': 'Sí', 'False': 'No', 'nan': 'No registrado'})


In [29]:
# --- Estatus de SDC ---
df['Estatus de SDC'] = df['Estatus de SDC'].astype(str).str.strip()
df['Estatus de SDC'] = df['Estatus de SDC'].replace('nan', 'No aplica')





In [31]:
# --- Mes desde Fecha ---
df.rename(columns={'ï»¿Fecha': 'Fecha'}, inplace=True)
df['Fecha'] = pd.to_datetime(df['Fecha'], errors='coerce', dayfirst=True)
meses_es = {1:'Enero',2:'Febrero',3:'Marzo',4:'Abril',5:'Mayo',6:'Junio',
            7:'Julio',8:'Agosto',9:'Septiembre',10:'Octubre',11:'Noviembre',12:'Diciembre'}
df['Mes'] = df['Fecha'].dt.month.map(meses_es).fillna('Sin fecha')

In [ ]:
# --- Hubo PDM (derivada de texto libre) ---
# Algunas columnas vienen con codificación "Â" / "Ã" al importarse desde CSV.
# Normalizamos el nombre de la columna para encontrarla sin depender del encoding exacto.
def clean_col_name(name):
    if pd.isna(name):
        return ""
    return (
        str(name)
        .lower()
        .replace("Ã", "")
        .replace("Â", "")
        .replace("¿", "")
        .replace("?", "")
        .replace(" ", "")
    )

col_pdm = next(
    (c for c in df.columns if "pruebademanejo" in clean_col_name(c) or "hubopruebademanejo" in clean_col_name(c)),
    None
)

if col_pdm is None:
    df["Hubo PDM"] = "No registrado"
else:
    df["Hubo PDM"] = df[col_pdm].astype(str).str.strip()
def clasificar_pdm(x):
    if x == 'nan' or pd.isna(x): return 'No registrado'
    if x.lower().startswith('sí hubo') or x.lower().startswith('si hubo'): return 'Sí hubo PDM'
    return 'No hubo PDM (con razón)'
df['Hubo PDM'] = df['Hubo PDM'].apply(clasificar_pdm)
